# Ensemble Judges: Multi-Judge Evaluation for Agents

This notebook builds on [LLM as a Judge Basics](LLM_as_Judge_Basics.ipynb) to explore **Ensemble Judge** evaluation—a powerful approach that leverages multiple judges to create more reliable, robust assessments of agent performance.

## Why Multiple Judges?

Just as you wouldn't trust a single reviewer to evaluate a complex research paper, relying on a single LLM judge has inherent limitations. Every model brings its own biases, inconsistencies, and blind spots to the evaluation process. By assembling multiple judges, we create a more balanced and trustworthy evaluation system that offers several key advantages:

**Reduced bias** comes from the fact that different models have different strengths and weaknesses. What one model might overlook, another catches. **Improved reliability** emerges when multiple judges reach consensus—their agreement gives us greater confidence in the assessment. **Diverse perspectives** mean we're evaluating responses from multiple angles, capturing nuances that a single judge might miss. Finally, **robustness** increases because the system becomes less vulnerable to individual judge failures or quirks.

## What You'll Learn

This notebook will guide you through the complete landscape of ensemble evaluation. You'll explore different ensemble evaluation strategies and understand when each approach makes sense. You'll learn how to implement multi-judge systems that coordinate multiple evaluators effectively. We'll dive into various aggregation methods—including mean, median, and weighted approaches—and help you choose the right one for your use case. You'll also discover how to measure and analyze agreement between judges, understanding what consensus (or disagreement) tells you about your agent's performance. Finally, you'll develop the judgment to know when ensemble approaches truly add value versus when a single judge suffices.

## Prerequisites

To get the most from this notebook, you should first complete [LLM as a Judge Basics](LLM_as_Judge_Basics.ipynb), which establishes the foundational concepts we'll build upon here.

# Setup

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    langgraph langchain langchain_ibm pandas matplotlib seaborn
! echo "::endgroup::"

In [ ]:
import json, time, requests, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Annotated, TypedDict
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.utils.utils import convert_to_secret_str
from langchain_core.messages import HumanMessage, AnyMessage, AIMessage
from langgraph.graph.state import CompiledStateGraph
from langgraph.graph.message import add_messages
from ibm_granite_community.notebook_utils import get_env_var

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✓ Libraries imported")

## Initialize Models

In [ ]:
# Agent model
llm_agent = init_chat_model(
    model="openai/gpt-oss-120b", model_provider="ibm",
    url=convert_to_secret_str(get_env_var("WATSONX_URL")),
    apikey=convert_to_secret_str(get_env_var("WATSONX_APIKEY")),
    project_id=get_env_var("WATSONX_PROJECT_ID"),
    params={"temperature": 0, "max_completion_tokens": 200, "repetition_penalty": 1.05}
)

# Multiple judge models with different temperatures
judges = {}
judge_model_ids = ["meta-llama/llama-3-3-70b-instruct", "mistralai/mistral-small-3-1-24b-instruct-2503", "ibm/granite-4-h-small"]
for id in judge_model_ids:
    judges[id] = init_chat_model(
        model=id, model_provider="ibm",
        url=convert_to_secret_str(get_env_var("WATSONX_URL")),
        apikey=convert_to_secret_str(get_env_var("WATSONX_APIKEY")),
        project_id=get_env_var("WATSONX_PROJECT_ID"),
        params={"temperature": 0, "max_completion_tokens": 500, "repetition_penalty": 1.05}
    )

print(f"✓ Initialized agent + {len(judges)} judges: {list(judges.keys())}")

## Create Test Agent

In [ ]:
# Tools and agent setup (reused from basics notebook)
AV_STOCK_API_KEY = convert_to_secret_str(get_env_var("AV_STOCK_API_KEY", "unset"))
WEATHER_API_KEY = convert_to_secret_str(get_env_var("WEATHER_API_KEY", "unset"))

def get_stock_price(ticker: str, date: str) -> dict:
    """
    Retrieves the lowest and highest stock prices for a given ticker and date.

    Args:
        ticker: The stock ticker symbol, for example, "IBM".
        date: The date in "YYYY-MM-DD" format.

    Returns:
        A dictionary containing the low and high stock prices.
    """
    apikey = AV_STOCK_API_KEY.get_secret_value()
    return {"low": "245.45", "high": "249.03"} if apikey == "unset" else {"low": "none", "high": "none"}

def get_current_weather(location: str) -> dict:
    """
    Fetches the current weather for a given location.

    Args:
        location: The city name.

    Returns:
        Weather information including temperature, description, and humidity.
    """
    apikey = WEATHER_API_KEY.get_secret_value()
    return {"description": "partly cloudy", "temperature": 22.5, "humidity": 65} if apikey == "unset" else {"description": "none", "temperature": "none", "humidity": "none"}

class State(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], add_messages]

tools=[get_stock_price, get_current_weather]
agent: CompiledStateGraph = create_agent(model=llm_agent)

def run_agent(graph, user_input: str) -> str:
    for event in graph.stream(State(messages=[HumanMessage(user_input)])):
        for value in event.values():
            msg = value["messages"][-1]
            if isinstance(msg, AIMessage) and not msg.tool_calls:
                return msg.content
    return ""

print("✓ Agent created")

# Data Structures

In [ ]:
@dataclass
class EvaluationScores:
    accuracy: float
    completeness: float
    helpfulness: float
    relevance: float
    overall: float
    justification: str
    raw_response: str

@dataclass
class EvaluationResult:
    user_query: str
    agent_response: str
    tool_results: Optional[str]
    scores: EvaluationScores
    evaluation_time: float
    judge_name: str = "judge"

@dataclass
class EnsembleEvaluationResult:
    user_query: str
    agent_response: str
    tool_results: Optional[str]
    individual_evaluations: List[EvaluationResult]
    aggregated_scores: EvaluationScores
    agreement_metrics: Dict[str, float]
    total_evaluation_time: float

print("✓ Data structures defined")

# Single Judge Evaluation (Review)

In [ ]:
def create_judge_prompt(user_query: str, agent_response: str, tool_results: Optional[str] = None) -> str:
    prompt = f"""You are an expert evaluator.
User Query: {user_query}
Agent Response: {agent_response}
"""
    if tool_results:
        prompt += f"Tool Results: {tool_results}\n"
    prompt += """\nEvaluate (0-10): Accuracy, Completeness, Helpfulness, Relevance
Format:
Scores:
- Accuracy: [0-10]
- Completeness: [0-10]
- Helpfulness: [0-10]
- Relevance: [0-10]
Overall Score: [0-10]
Justification: [brief explanation]"""
    return prompt

def parse_judge_response(response: str) -> EvaluationScores:
    scores = {}
    justification = ""
    for line in response.split('\n'):
        if ':' in line:
            key, value = line.split(':', 1)
            key, value = key.strip().lower(), value.strip()
            try:
                score = float(value.split('/')[0].split()[0])
                if 'accuracy' in key: scores['accuracy'] = score
                elif 'completeness' in key: scores['completeness'] = score
                elif 'helpfulness' in key: scores['helpfulness'] = score
                elif 'relevance' in key: scores['relevance'] = score
                elif 'overall' in key: scores['overall'] = score
                elif 'justification' in key: justification = value
            except: pass
    return EvaluationScores(
        accuracy=scores.get('accuracy', 5.0), completeness=scores.get('completeness', 5.0),
        helpfulness=scores.get('helpfulness', 5.0), relevance=scores.get('relevance', 5.0),
        overall=scores.get('overall', 5.0), justification=justification or "N/A", raw_response=response
    )

def evaluate_with_judge(user_query, agent_response, tool_results=None, judge_model=None, judge_name="judge"):
    start = time.time()
    response = judge_model.invoke(create_judge_prompt(user_query, agent_response, tool_results))
    return EvaluationResult(
        user_query, agent_response, tool_results,
        parse_judge_response(response.content if hasattr(response, 'content') else str(response)),
        time.time() - start, judge_name
    )

print("✓ Single judge evaluation ready")

# Ensemble Strategies

In [ ]:
def calculate_agreement_metrics(evaluations: List[EvaluationResult]) -> Dict[str, float]:
    """Calculate agreement metrics across judges."""
    if len(evaluations) < 2:
        return {"note": "Need 2+ judges"}
    
    dimensions = ['accuracy', 'completeness', 'helpfulness', 'relevance', 'overall']
    metrics = {}
    
    for dim in dimensions:
        scores = [getattr(e.scores, dim) for e in evaluations]
        mean = sum(scores) / len(scores)
        std = (sum((x - mean)**2 for x in scores) / len(scores))**0.5
        metrics[f"{dim}_mean"] = mean
        metrics[f"{dim}_std"] = std
        metrics[f"{dim}_range"] = max(scores) - min(scores)
    
    avg_std = sum(metrics[f"{d}_std"] for d in dimensions) / len(dimensions)
    metrics["overall_agreement"] = max(0, 10 - avg_std)
    return metrics

def aggregate_scores_mean(evaluations: List[EvaluationResult]) -> EvaluationScores:
    """Aggregate using mean."""
    dims = ['accuracy', 'completeness', 'helpfulness', 'relevance', 'overall']
    agg = {d: sum(getattr(e.scores, d) for e in evaluations) / len(evaluations) for d in dims}
    just = " | ".join(f"{e.judge_name}: {e.scores.justification}" for e in evaluations)
    return EvaluationScores(**agg, justification=just, raw_response="Mean aggregation")

def aggregate_scores_median(evaluations: List[EvaluationResult]) -> EvaluationScores:
    """Aggregate using median (robust to outliers)."""
    dims = ['accuracy', 'completeness', 'helpfulness', 'relevance', 'overall']
    agg = {}
    for d in dims:
        scores = sorted([getattr(e.scores, d) for e in evaluations])
        n = len(scores)
        agg[d] = scores[n//2] if n % 2 else (scores[n//2-1] + scores[n//2]) / 2
    just = " | ".join(f"{e.judge_name}: {e.scores.justification}" for e in evaluations)
    return EvaluationScores(**agg, justification=just, raw_response="Median aggregation")

def aggregate_scores_weighted(evaluations: List[EvaluationResult], weights: List[float]) -> EvaluationScores:
    """Aggregate using weighted average."""
    if len(weights) != len(evaluations) or abs(sum(weights) - 1.0) > 0.01:
        raise ValueError("Weights must match evaluations and sum to 1.0")
    dims = ['accuracy', 'completeness', 'helpfulness', 'relevance', 'overall']
    agg = {d: sum(getattr(e.scores, d) * w for e, w in zip(evaluations, weights)) for d in dims}
    just = " | ".join(f"{e.judge_name}(w={w:.2f}): {e.scores.justification}" for e, w in zip(evaluations, weights))
    return EvaluationScores(**agg, justification=just, raw_response="Weighted aggregation")

print("✓ Aggregation strategies ready")

# Ensemble Evaluation

In [ ]:
def evaluate_with_ensemble(
    user_query: str, agent_response: str, tool_results: Optional[str] = None,
    judge_models: Optional[Dict] = None, aggregation_method: str = "mean",
    weights: Optional[List[float]] = None, parallel: bool = True
) -> EnsembleEvaluationResult:
    """Evaluate using ensemble of judges."""
    if judge_models is None:
        judge_models = judges
    
    start = time.time()
    individual_evaluations = []
    
    if parallel:
        with ThreadPoolExecutor(max_workers=len(judge_models)) as executor:
            futures = {executor.submit(evaluate_with_judge, user_query, agent_response, 
                                      tool_results, model, name): name 
                      for name, model in judge_models.items()}
            for future in as_completed(futures):
                individual_evaluations.append(future.result())
    else:
        for name, model in judge_models.items():
            individual_evaluations.append(
                evaluate_with_judge(user_query, agent_response, tool_results, model, name)
            )
    
    # Aggregate
    if aggregation_method == "mean":
        aggregated = aggregate_scores_mean(individual_evaluations)
    elif aggregation_method == "median":
        aggregated = aggregate_scores_median(individual_evaluations)
    elif aggregation_method == "weighted":
        aggregated = aggregate_scores_weighted(individual_evaluations, weights)
    else:
        raise ValueError(f"Unknown method: {aggregation_method}")
    
    return EnsembleEvaluationResult(
        user_query, agent_response, tool_results, individual_evaluations,
        aggregated, calculate_agreement_metrics(individual_evaluations), time.time() - start
    )

print("✓ Ensemble evaluation ready")

# Example: Weather Query

In [ ]:
query = "What is the weather in Miami?"
response = run_agent(agent, query)
print(f"Query: {query}\nResponse: {response}\n")

result = evaluate_with_ensemble(
    user_query=query, agent_response=response,
    tool_results="get_current_weather('Miami') -> {description: 'partly cloudy', temp: 22.5}",
    aggregation_method="mean", parallel=True
)

print("="*60)
print("ENSEMBLE EVALUATION")
print("="*60)
print("\nIndividual Judges:")
for e in result.individual_evaluations:
    print(f"  {e.judge_name}: Overall={e.scores.overall:.1f}/10")

print(f"\nAggregated (Mean):")
print(f"  Accuracy: {result.aggregated_scores.accuracy:.2f}/10")
print(f"  Completeness: {result.aggregated_scores.completeness:.2f}/10")
print(f"  Helpfulness: {result.aggregated_scores.helpfulness:.2f}/10")
print(f"  Relevance: {result.aggregated_scores.relevance:.2f}/10")
print(f"  Overall: {result.aggregated_scores.overall:.2f}/10")

print(f"\nAgreement: {result.agreement_metrics.get('overall_agreement', 0):.2f}/10")
print(f"Time: {result.total_evaluation_time:.2f}s")
print("="*60)

# Comparing Aggregation Methods

In [ ]:
query2 = "Tell me about Apple"
response2 = run_agent(agent, query2)

methods = ["mean", "median"]
results = {}

for method in methods:
    results[method] = evaluate_with_ensemble(
        user_query=query2, agent_response=response2,
        tool_results="No tools called", aggregation_method=method
    )

print("\nCOMPARISON OF AGGREGATION METHODS")
print("="*60)
data = []
for method, r in results.items():
    data.append({
        "Method": method.capitalize(),
        "Accuracy": f"{r.aggregated_scores.accuracy:.2f}",
        "Completeness": f"{r.aggregated_scores.completeness:.2f}",
        "Overall": f"{r.aggregated_scores.overall:.2f}",
        "Agreement": f"{r.agreement_metrics.get('overall_agreement', 0):.2f}"
    })

df = pd.DataFrame(data)
print(df.to_string(index=False))
print("="*60)

# Conclusion

## Key Takeaways

✅ **Ensemble judges reduce bias** and improve reliability <br/>
✅ **Multiple aggregation methods** suit different needs <br/>
✅ **Agreement metrics** indicate evaluation confidence <br/>
✅ **Parallel execution** reduces latency <br/>

## When to Use Ensemble Judges

**Use ensemble when:**
- High-stakes decisions require confidence
- Single judge shows inconsistency
- You need diverse evaluation perspectives
- Budget allows for multiple evaluations

**Stick with single judge when:**
- Budget/latency constraints are tight
- Evaluation is low-stakes
- Single judge performs consistently